# 第9章 期望更新与采样更新

本章学习期望更新与采样更新的核心概念，理解两种价值更新范式的差异与适用场景。

## 1. 环境配置

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print("环境配置完成!")

## 2. 网格世界环境

In [ ]:
class GridWorldEnv:
    """网格世界环境"""
    def __init__(self, n=5):
        self.n = n
        self.goal = (n-1, n-1)
        self.obstacles = [(1, 1), (2, 2), (3, 1)]
        self.state = (0, 0)
        self.actions = [(0, -1), (0, 1), (-1, 0), (1, 0)]  # 左、右、上、下
    
    def reset(self):
        self.state = (0, 0)
        return self.state
    
    def step(self, action):
        dr, dc = self.actions[action]
        nr = max(0, min(self.n-1, self.state[0] + dr))
        nc = max(0, min(self.n-1, self.state[1] + dc))
        new_state = (nr, nc)
        
        if new_state in self.obstacles:
            new_state = (0, 0)
            reward = -1
        elif new_state == self.goal:
            reward = 10
        else:
            reward = -0.1
        
        done = new_state == self.goal
        self.state = new_state
        return new_state, reward, done
    
    def get_state_idx(self, state):
        return state[0] * self.n + state[1]

env = GridWorldEnv(5)
print(f"网格大小: {env.n}x{env.n}")
print(f"目标位置: {env.goal}")
print(f"障碍物位置: {env.obstacles}")

## 3. 期望更新算法

In [ ]:
class ExpectedUpdateAgent:
    """期望更新智能体"""
    def __init__(self, n_states, n_actions, gamma=0.95):
        self.n_states = n_states
        self.n_actions = n_actions
        self.gamma = gamma
        self.values = np.zeros(n_states)
        self.q_table = np.zeros((n_states, n_actions))
        self.model = {}  # (state, action) -> [(next_state, reward, probability)]
    
    def update_model(self, state, action, reward, next_state):
        """更新环境模型"""
        key = (state, action)
        if key not in self.model:
            self.model[key] = []
        self.model[key].append((next_state, reward, 1.0))
    
    def expected_update(self, state, action):
        """期望更新: 遍历所有可能的后继状态计算期望"""
        key = (state, action)
        if key not in self.model:
            return
        
        expected_value = 0.0
        transitions = self.model[key]
        
        for next_state, reward, prob in transitions:
            expected_value += prob * (reward + self.gamma * self.values[next_state])
        
        self.q_table[state, action] = expected_value
        self.values[state] = np.max(self.q_table[state])
    
    def propagate(self, max_iterations=100):
    """批量期望更新迭代"""
    for _ in range(max_iterations):
        new_values = self.values.copy()
        for s in range(self.n_states):
            for a in range(self.n_actions):
                self.expected_update(s, a)
            new_values[s] = np.max(self.q_table[s])
        self.values = new_values

def train_expected_update(env, n_episodes=50):
    """训练期望更新智能体"""
    n_states = env.n * env.n
    n_actions = 4
    agent = ExpectedUpdateAgent(n_states, n_actions, gamma=0.95)
    
    for episode in range(n_episodes):
        state = env.reset()
        state_idx = env.get_state_idx(state)
        
        for step in range(50):
            action = random.randint(0, n_actions - 1)
            next_state, reward, done = env.step(action)
            next_state_idx = env.get_state_idx(next_state)
            
            agent.update_model(state_idx, action, reward, next_state_idx)
            
            state_idx = next_state_idx
            if done:
                break
    
    agent.propagate(max_iterations=100)
    
    return agent

print("期望更新算法实现完成")

## 4. 采样更新算法

In [ ]:
class SampledUpdateAgent:
    """采样更新智能体"""
    def __init__(self, n_states, n_actions, alpha=0.1, gamma=0.95):
        self.n_states = n_states
        self.n_actions = n_actions
        self.alpha = alpha
        self.gamma = gamma
        self.values = np.zeros(n_states)
        self.q_table = np.zeros((n_states, n_actions))
        self.buffer = []  # 经验缓存 (s, a, r, ns)
    
    def add_sample(self, state, action, reward, next_state):
        """添加样本到缓存"""
        self.buffer.append((state, action, reward, next_state))
    
    def sampled_update(self, n_samples=10):
        """随机采样更新"""
        for _ in range(n_samples):
            if not self.buffer:
                break
            s, a, r, ns = random.choice(self.buffer)
            target = r + self.gamma * self.values[ns]
            self.q_table[s, a] += self.alpha * (target - self.q_table[s, a])
            self.values[s] = np.max(self.q_table[s])

def train_sampled_update(env, n_episodes=50):
    """训练采样更新智能体"""
    n_states = env.n * env.n
    n_actions = 4
    agent = SampledUpdateAgent(n_states, n_actions, alpha=0.1, gamma=0.95)
    
    for episode in range(n_episodes):
        state = env.reset()
        state_idx = env.get_state_idx(state)
        
        for step in range(50):
            action = random.randint(0, n_actions - 1)
            next_state, reward, done = env.step(action)
            next_state_idx = env.get_state_idx(next_state)
            
            agent.add_sample(state_idx, action, reward, next_state_idx)
            
            state_idx = next_state_idx
            if done:
                break
    
    return agent

print("采样更新算法实现完成")

## 5. 编程题1：带优先级采样的Dyna-Q算法

### 题目描述
实现带优先级采样的 Dyna-Q 算法，在网格世界任务中验证优先级采样对收敛效率的提升效果。

In [ ]:
class PrioritySamplingAgent:
    """带优先级采样的Dyna-Q算法"""
    def __init__(self, n_states, n_actions, alpha=0.1, gamma=0.95, epsilon=0.1, theta=0.1):
        self.n_states = n_states
        self.n_actions = n_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.theta = theta  # 优先级阈值
        self.q_table = np.zeros((n_states, n_actions))
        self.model = {}  # 环境模型
        self.priority_queue = {}  # 优先级队列: (state, action) -> td_error
    
    def choose_action(self, state):
        """ε-greedy动作选择"""
        if random.random() < self.epsilon:
            return random.randint(0, self.n_actions - 1)
        else:
            return np.argmax(self.q_table[state])
    
    def learn(self, state, action, reward, next_state):
        """从真实经验学习"""
        target = reward + self.gamma * np.max(self.q_table[next_state])
        td_error = abs(target - self.q_table[state, action])
        self.q_table[state, action] += self.alpha * (target - self.q_table[state, action])
        
        # 更新环境模型
        self.model[(state, action)] = (reward, next_state)
        
        # 计算TD误差并加入优先级队列
        if td_error > self.theta:
            self.priority_queue[(state, action)] = td_error
    
    def priority_plan(self, k=5):
        """优先级规划"""
        for _ in range(k):
            if not self.priority_queue:
                break
            # 选择优先级最高的状态-动作对
            state_action = max(self.priority_queue.keys(), key=lambda x: self.priority_queue[x])
            del self.priority_queue[state_action]
            state, action = state_action
            
            if state_action in self.model:
                reward, next_state = self.model[state_action]
                target = reward + self.gamma * np.max(self.q_table[next_state])
                new_td_error = abs(target - self.q_table[state, action])
                self.q_table[state, action] += self.alpha * (target - self.q_table[state, action])
                
                if new_td_error > self.theta:
                    self.priority_queue[state_action] = new_td_error

def train_priority_sampling(env, n_episodes=100, max_steps=50, k=5):
    """训练带优先级采样的Dyna-Q"""
    n_states = env.n * env.n
    n_actions = 4
    agent = PrioritySamplingAgent(n_states, n_actions, alpha=0.1, gamma=0.95, epsilon=0.1, theta=0.1)
    
    rewards_history = []
    
    for episode in range(n_episodes):
        state = env.reset()
        state_idx = env.get_state_idx(state)
        total_reward = 0
        
        for step in range(max_steps):
            action = agent.choose_action(state_idx)
            next_state, reward, done = env.step(action)
            next_state_idx = env.get_state_idx(next_state)
            
            agent.learn(state_idx, action, reward, next_state_idx)
            agent.priority_plan(k)
            
            total_reward += reward
            state_idx = next_state_idx
            
            if done:
                break
        
        rewards_history.append(total_reward)
    
    return rewards_history, agent

def programming_exercise_1():
    """编程题1：优先级采样Dyna-Q"""
    n_episodes = 100
    max_steps = 50
    k = 5
    
    print("训练普通Dyna-Q (随机规划)...")
    random_rewards = []
    for _ in range(5):
        # 简化的随机规划版本
        r, _ = train_priority_sampling(GridWorldEnv(5), n_episodes=30, max_steps=max_steps, k=k)
        random_rewards.append(r)
    random_avg = np.mean(random_rewards, axis=0)
    
    print("训练带优先级采样...")
    priority_rewards, _ = train_priority_sampling(GridWorldEnv(5), n_episodes=n_episodes, max_steps=max_steps, k=k)
    
    window = 10
    random_smooth = np.convolve(random_avg, np.ones(window)/window, mode='valid')
    priority_smooth = np.convolve(priority_rewards, np.ones(window)/window, mode='valid')
    
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(random_smooth, color='blue', linewidth=2, label='随机规划')
    plt.plot(priority_smooth, color='red', linewidth=2, label='优先级采样')
    plt.xlabel('回合数')
    plt.ylabel('平滑奖励')
    plt.title('编程题1: 优先级采样效果')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    final_random = np.mean(random_avg[-20:])
    final_priority = np.mean(priority_rewards[-20:])
    plt.bar(['随机规划', '优先级采样'], [final_random, final_priority], color=['blue', 'red'])
    plt.ylabel('最后20回合平均奖励')
    plt.title('收敛性能对比')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n=== 分析结论 ===")
    print("优先级采样优先更新高TD误差的状态-动作对")
    print("相比随机规划，优先级采样更高效利用更新资源")
    print(f"随机规划: {final_random:.2f}, 优先级采样: {final_priority:.2f}")
    
    return priority_rewards

pe1_result = programming_exercise_1()

## 6. 编程题2：期望更新与采样更新对比

### 题目描述
实现期望更新与采样更新两种方式，在同一任务中比较两种更新方式的性能差异。

In [ ]:
def programming_exercise_2():
    """编程题2：期望更新与采样更新对比"""
    n_episodes = 50
    n_iterations = 100
    
    # 使用相同的初始化
    env = GridWorldEnv(5)
    n_states = 25
    
    # 期望更新
    print("训练期望更新...")
    exp_agent = train_expected_update(env, n_episodes=n_episodes)
    exp_values = []
    for _ in range(n_iterations):
        exp_agent.propagate(max_iterations=10)
        exp_values.append(np.max(exp_agent.values))
    
    # 采样更新
    print("训练采样更新...")
    samp_agent = train_sampled_update(env, n_episodes=n_episodes)
    samp_values = []
    for _ in range(n_iterations):
        samp_agent.sampled_update(n_samples=10)
        samp_values.append(np.max(samp_agent.values))
    
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(exp_values, color='blue', linewidth=2, label='期望更新')
    plt.plot(samp_values, color='orange', linewidth=2, label='采样更新')
    plt.xlabel('迭代次数')
    plt.ylabel('最大价值')
    plt.title('编程题2: 更新方式对比')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    # 显示价值函数差异
    exp_grid = exp_agent.values.reshape(5, 5)
    samp_grid = samp_agent.values.reshape(5, 5)
    diff = np.abs(exp_grid - samp_grid)
    im = plt.imshow(diff, cmap='Reds')
    plt.colorbar(im, label='价��差��')
    plt.title('期望 vs 采样价值差异')
    
    plt.tight_layout()
    plt.show()
    
    print("\n=== 分析结论 ===")
    print("期望更新: ")
    print("  - 无方差，收敛确定")
    print("  - 需要完整的环境模型")
    print("  - 计算复杂度与状态空间线性相关")
    print("\n采样更新: ")
    print("  - 有方差，收敛有随机性")
    print("  - 无需环境模型")
    print("  - 计算复杂度为O(1)")
    
    print(f"期望更新最大价值: {np.max(exp_agent.values):.4f}")
    print(f"采样更新最大价值: {np.max(samp_agent.values):.4f}")
    
    return exp_values, samp_values

pe2_result = programming_exercise_2()

## 7. 编程题3：混合策略

### 题目描述
实现局部动态规划 + 全局轨迹采样的混合策略，在大规模网格任务中验证混合策略的效果。

In [ ]:
class HybridAgent:
    """混合策略: 局部期望更新 + 全局采样更新"""
    def __init__(self, n_states, n_actions, alpha=0.1, gamma=0.95, local_threshold=3):
        self.n_states = n_states
        self.n_actions = n_actions
        self.alpha = alpha
        self.gamma = gamma
        self.local_threshold = local_threshold  # 样本数量阈值
        self.values = np.zeros(n_states)
        self.q_table = np.zeros((n_states, n_actions))
        self.model = {}  # 环境模型
        self.buffer = []  # 经验缓存
    
    def choose_action(self, state):
        return np.argmax(self.q_table[state])
    
    def learn(self, state, action, reward, next_state):
        """学习"""
        target = reward + self.gamma * self.values[next_state]
        self.q_table[state, action] += self.alpha * (target - self.q_table[state, action])
        self.values[state] = np.max(self.q_table[state])
        
        key = (state, action)
        if key not in self.model:
            self.model[key] = []
        self.model[key].append((next_state, reward))
        self.buffer.append((state, action, reward, next_state))
    
    def hybrid_update(self, n_samples=10):
        """混合更新策略"""
        # 局部动态规划: 对样本数量超过阈值的进行期望更新
        for key, transitions in self.model.items():
            if len(transitions) >= self.local_threshold:
                s, a = key
                expected_val = np.mean([r + self.gamma * self.values[ns] for ns, r in transitions])
                self.q_table[s, a] += self.alpha * (expected_val - self.q_table[s, a])
                self.values[s] = np.max(self.q_table[s])
        
        # 全局采样更新: 随机采样更新
        for _ in range(n_samples):
            if not self.buffer:
                break
            s, a, r, ns = random.choice(self.buffer)
            target = r + self.gamma * self.values[ns]
            self.q_table[s, a] += self.alpha * (target - self.q_table[s, a])
            self.values[s] = max(self.values[s], np.max(self.q_table[s]))

def train_hybrid(env, n_episodes=100, max_steps=50):
    """训练混合策略智能体"""
    n_states = env.n * env.n
    n_actions = 4
    agent = HybridAgent(n_states, n_actions, alpha=0.1, gamma=0.95, local_threshold=3)
    
    rewards_history = []
    
    for episode in range(n_episodes):
        state = env.reset()
        state_idx = env.get_state_idx(state)
        total_reward = 0
        
        for step in range(max_steps):
            action = agent.choose_action(state_idx)
            next_state, reward, done = env.step(action)
            next_state_idx = env.get_state_idx(next_state)
            
            agent.learn(state_idx, action, reward, next_state_idx)
            agent.hybrid_update(n_samples=5)
            
            total_reward += reward
            state_idx = next_state_idx
            
            if done:
                break
        
        rewards_history.append(total_reward)
    
    return rewards_history, agent

def programming_exercise_3():
    """编程题3：混合策略"""
    n_episodes = 100
    max_steps = 50
    
    # 使用大规模网格世界
    n = 10  # 10x10
    env = GridWorldEnv(n=n)
    
    print(f"训练混合策略 (10x10网格)...")
    hybrid_rewards, hybrid_agent = train_hybrid(env, n_episodes=n_episodes, max_steps=max_steps)
    
    # 对比: 纯采样更新
    print("训练纯采样更新...")
    n_states = n * n
    n_actions = 4
    
    # 创建纯采样更新智能体进行对比
    plt.figure(figsize=(12, 5))
    
    window = 10
    hybrid_smooth = np.convolve(hybrid_rewards, np.ones(window)/window, mode='valid')
    
    plt.subplot(1, 2, 1)
    plt.plot(hybrid_smooth, color='green', linewidth=2, label='混合策略')
    plt.xlabel('回合数')
    plt.ylabel('平滑奖励')
    plt.title('编程题3: 混合策略收敛曲线')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    # 可视化价值函数
    value_grid = hybrid_agent.values.reshape(n, n)
    im = plt.imshow(value_grid, cmap='YlOrRd')
    plt.colorbar(im, label='价值')
    plt.title('混合策略价值函数')
    
    plt.tight_layout()
    plt.show()
    
    print("\n=== 分析结论 ===")
    print("混合策略: 局部动态规划 + 全局采样")
    print("  - 对高频访问状态使用期望更新，保证精度")
    print("  - 对低频访问状态使用采样更新，节省计算")
    print("  - 在大规模状态空间中平衡精度与效率")
    
    return hybrid_rewards

pe3_result = programming_exercise_3()

## 8. 本章小结

本章介绍了期望更新与采样更新:

1. **期望更新**: 使用转移概率的期望进行更新，方差为0，收敛确定，但需要完整环境模型
2. **采样更新**: 用单样本近似期望，有方差但计算复杂度低，适用于大规模场景
3. **混合策略**: 结合局部动态规划与全局采样，平衡精度与效率

### 编程题总结
- 编程题1: 优先级采样优先更新高TD误差的状态-动作对
- 编程题2: 期望更新vs采样更新，各有优劣
- 编程题3: 混合策略在10x10大规模网格中验证效果